## <b> Construir un spread vertical de calls y distinguir su payoff de su ganancia. </b>

Selecciona dos calls de la misma fecha y vencimiento, con strikes $ K_1 < K_2 $ . Compras la de strike bajo y vendes la de strike alto. Así estudias una posición que se beneficia de una subida, pero tiene una ganancia limitada.

El coste usando las cotizaciones sería:

$ D = \text{ask}(K_1) - \text{bid}(K_2) $

Y su resultado al vencimiento, por acción y antes de costes:

$ \Pi(S_T) = \max(S_T - K_1, 0) - \max(S_T - K_2, 0) - D $ 

**Tu pregunta:** ¿qué sacrificas y qué consigues al vender la segunda call en lugar de comprar únicamente la primera?

Resuélvelo en cuatro pasos:

1. Selecciona dos contratos y explica por qué compras uno y vendes el otro.
2. Calcula el coste del spread y compáralo con el coste de comprar solo la primera call.
3. Calcula ambos resultados para varios precios hipotéticos de SPY al vencimiento: por debajo de $ K_1 $ , entre los strikes y por encima de $ K_2$ .
4. Grafica ambas curvas e identifica pérdida máxima, ganancia máxima y punto de equilibrio del spread.

**El aprendizaje práctico:** reducir la prima inicial tiene una contrapartida: renuncias a parte de la ganancia si SPY sube mucho. Estar ITM tampoco garantiza haber recuperado lo que pagaste.


In [1]:
from pathlib import Path
from collections import defaultdict
from datetime import date
import gzip
import json
import pandas as pd

# Lista editable para el ejercicio.
exercise_dates = ['2020-01-02', '2022-10-03', '2025-04-03']

cwd = Path.cwd().resolve()
repo_root = next((parent for parent in (cwd, *cwd.parents) if (parent / 'quant-research-notes').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('No se encontró la raíz del repositorio: se esperaba una carpeta quant-research-notes.')

hmm_root = repo_root / 'quant-research-notes' / 'HMM_regime_options'
raw_dir = hmm_root / 'data' / 'raw'
price_path = hmm_root / 'data' / 'derived' / 'spy_daily_adjusted_2020_2026.csv'
if not raw_dir.is_dir():
    raise FileNotFoundError(f'No existe el directorio de cadenas originales: {raw_dir}')
if not price_path.is_file():
    raise FileNotFoundError(f'No existe la tabla local de cierres SPY: {price_path}')

prices = pd.read_csv(price_path, parse_dates=['date']).set_index('date')
if 'close' not in prices.columns:
    raise KeyError(f'Falta el campo close en {price_path}')

def load_chain(chain_date):
    candidates = [raw_dir / f'spy_historical_options_{chain_date}.json.gz', raw_dir / f'spy_historical_options_{chain_date}.json']
    chain_path = next((path for path in candidates if path.is_file()), None)
    if chain_path is None:
        raise FileNotFoundError(f'Falta la cadena original de SPY para {chain_date}; se buscó .json.gz y .json en {raw_dir}')
    opener = gzip.open if chain_path.suffix == '.gz' else open
    with opener(chain_path, 'rt', encoding='utf-8') as handle:
        payload = json.load(handle)
    records = payload.get('data')
    if not isinstance(records, list):
        raise KeyError(f'La cadena {chain_path.name} no contiene una lista data de contratos')
    return chain_path, records

work_frames = []
required_fields = {'contractID', 'type', 'expiration', 'strike', 'bid', 'ask'}
for chain_date in exercise_dates:
    observation_date = pd.Timestamp(chain_date)
    if observation_date not in prices.index:
        raise KeyError(f'No hay cierre SPY para {chain_date} en {price_path.name}')
    spy_close = prices.loc[observation_date, 'close']
    chain_path, raw_records = load_chain(chain_date)

    grouped = defaultdict(list)
    for record in raw_records:
        missing = required_fields.difference(record)
        if missing:
            raise KeyError(f'Faltan campos {sorted(missing)} en la cadena {chain_path.name}; contractID={record.get("contractID")}')
        grouped[str(record['contractID'])].append(record)
    deduplicated = []
    for contract_id, records in grouped.items():
        fingerprints = {json.dumps(record, sort_keys=True, separators=(',', ':'), ensure_ascii=False) for record in records}
        if len(fingerprints) > 1:
            raise RuntimeError(f'Conflicto: contractID {contract_id} tiene registros distintos en {chain_path.name}; el ejercicio se detuvo.')
        deduplicated.append(records[0])

    expiration_sides = defaultdict(set)
    for record in deduplicated:
        try:
            expiration = date.fromisoformat(str(record['expiration']))
            dte = (expiration - observation_date.date()).days
        except ValueError as exc:
            raise ValueError(f'Vencimiento inválido en {chain_path.name}: {record["expiration"]}') from exc
        if 27 <= dte <= 33 and record['type'] in {'call', 'put'}:
            expiration_sides[str(record['expiration'])].add(record['type'])
    common_expirations = [expiration for expiration, sides in expiration_sides.items() if sides == {'call', 'put'}]
    if not common_expirations:
        raise ValueError(f'No hay vencimiento común call/put entre 27 y 33 DTE para {chain_date} en {chain_path.name}')
    selected_expiration = min(
        common_expirations,
        key=lambda expiration: (abs((date.fromisoformat(expiration) - observation_date.date()).days - 30), expiration),
    )
    selected_dte = (date.fromisoformat(selected_expiration) - observation_date.date()).days
    rows = []
    for record in deduplicated:
        if record['type'] not in {'call', 'put'} or str(record['expiration']) != selected_expiration:
            continue
        try:
            strike = float(record['strike'])
        except (TypeError, ValueError) as exc:
            raise ValueError(f'Strike inválido para {record["contractID"]} en {chain_path.name}') from exc
        if abs(strike / float(spy_close) - 1.0) > 0.05:
            continue
        bid_number = pd.to_numeric(record['bid'], errors='coerce')
        ask_number = pd.to_numeric(record['ask'], errors='coerce')
        rows.append({
            'date': chain_date, 'type': record['type'], 'expiration': selected_expiration, 'DTE': selected_dte,
            'strike': strike, 'SPY_close': float(spy_close), 'bid': record['bid'], 'ask': record['ask'],
            'mid': (bid_number + ask_number) / 2, 'contractID': record['contractID'],
        })
    work_frames.append(pd.DataFrame(rows))

options_work = pd.concat(work_frames, ignore_index=True)
options_work = options_work[['date', 'type', 'expiration', 'DTE', 'strike', 'SPY_close', 'bid', 'ask', 'mid', 'contractID']]
options_work = options_work.sort_values(['date', 'type', 'strike', 'contractID']).reset_index(drop=True)
display(options_work)
print('Filas por fecha:')
display(options_work.groupby('date').size().rename('rows').reset_index())

,date,type,expiration,DTE,strike,SPY_close,bid,ask,mid,contractID
0,2020-01-02,call,2020-01-31,29,309.0,324.87,17.19,17.30,17.245,SPY200131C00309000
1,2020-01-02,call,2020-01-31,29,309.5,324.87,16.73,16.87,16.800,SPY200131C00309500
2,2020-01-02,call,2020-01-31,29,310.0,324.87,16.27,16.39,16.330,SPY200131C00310000
3,2020-01-02,call,2020-01-31,29,310.5,324.87,15.81,15.94,15.875,SPY200131C00310500
4,2020-01-02,call,2020-01-31,29,311.0,324.87,15.35,15.48,15.415,SPY200131C00311000
...,...,...,...,...,...,...,...,...,...,...
261,2025-04-03,put,2025-05-02,29,560.0,536.70,24.81,27.17,25.990,SPY250502P00560000
262,2025-04-03,put,2025-05-02,29,561.0,536.70,25.49,28.79,27.140,SPY250502P00561000
263,2025-04-03,put,2025-05-02,29,562.0,536.70,26.18,29.42,27.800,SPY250502P00562000
264,2025-04-03,put,2025-05-02,29,562.5,536.70,26.53,29.74,28.135,SPY250502P00562500


Filas por fecha:


,date,rows
0,2020-01-02,112
1,2022-10-03,70
2,2025-04-03,84


## Paso 1 — Elegir los dos contratos y definir la posición

Usaremos la cadena archivada de SPY del **2025-04-03**. El cierre sin ajustar de SPY ese día fue **536.70**. Elegimos el vencimiento común más cercano a 30 días: **2025-05-02**, a **29 DTE**.

| Papel | Contrato | Strike | Bid | Ask | Delta |
|---|---|---:|---:|---:|---:|
| Comprar | `SPY250502C00535000` | 535 | 18.11 | 18.19 | 0.54982 |
| Vender | `SPY250502C00540000` | 540 | 15.10 | 15.18 | 0.49984 |

Construimos un **spread vertical alcista de calls**:

- Compramos la call de strike bajo, \(K_1=535\). Da el derecho a comprar SPY a 535 al vencimiento.
- Vendemos la call de strike alto, \(K_2=540\). Nos obliga a vender SPY a 540 si quien la posee decide ejercer.
- Ambas opciones son calls de SPY, tienen el mismo vencimiento y cumplen \(K_1 < K_2\).

La call comprada es la más cercana al dinero: su strike está apenas por debajo del cierre de SPY de 536.70. La call vendida tiene un strike superior, por lo que aporta una prima que reduce el costo inicial de la posición.

A cambio de reducir ese costo, aceptamos que, si SPY termina muy por encima de 540, la call vendida compensa la ganancia adicional de la call comprada. Por tanto, el spread conserva exposición a una subida moderada, pero impone un techo a su ganancia.

En el siguiente paso usaremos el **ask de la call comprada** y el **bid de la call vendida**: son las cotizaciones relevantes para representar una entrada conservadora y potencialmente ejecutable.

In [2]:
# Paso 1: seleccionar y documentar las dos calls del spread vertical.
# La selección usa cotizaciones archivadas; todavía no calcula coste ni payoff.

fecha = "2025-04-03"
contrato_largo = "SPY250502C00535000"   # call comprada: K1 = 535
contrato_corto = "SPY250502C00540000"   # call vendida:   K2 = 540

spread_calls = (
    options_work.loc[
        options_work["contractID"].isin([contrato_largo, contrato_corto])
    ]
    .copy()
    .sort_values("strike")
    .reset_index(drop=True)
)

# Controles mínimos: dos calls, misma fecha/vencimiento y strikes crecientes.
if len(spread_calls) != 2:
    raise ValueError("No se encontraron exactamente los dos contratos seleccionados.")

if (
    (spread_calls["type"] != "call").any()
    or spread_calls["date"].nunique() != 1
    or spread_calls["expiration"].nunique() != 1
    or not spread_calls["strike"].is_monotonic_increasing
):
    raise ValueError(
        "El par no cumple: dos calls, misma fecha, mismo vencimiento y K1 < K2."
    )

# La call de strike bajo se compra; la de strike alto se vende.
spread_calls["posición"] = ["Comprar call (K1)", "Vender call (K2)"]

columnas = [
    "posición", "date", "expiration", "DTE", "SPY_close",
    "contractID", "strike", "bid", "ask", "mid", "delta",
]

# `delta` no estaba en options_work inicial; mostrarla solo si se añadió.
columnas = [col for col in columnas if col in spread_calls.columns]

display(spread_calls[columnas])

print(
    f"Spread elegido: comprar K={spread_calls.loc[0, 'strike']:.0f} "
    f"y vender K={spread_calls.loc[1, 'strike']:.0f}, "
    f"con vencimiento {spread_calls.loc[0, 'expiration']}."
)

,posición,date,expiration,DTE,SPY_close,contractID,strike,bid,ask,mid
0,Comprar call (K1),2025-04-03,2025-05-02,29,536.7,SPY250502C00535000,535.0,18.11,18.19,18.15
1,Vender call (K2),2025-04-03,2025-05-02,29,536.7,SPY250502C00540000,540.0,15.10,15.18,15.14


Spread elegido: comprar K=535 y vender K=540, con vencimiento 2025-05-02.


## Paso 2 — Calcular y comparar el coste inicial

Para abrir el spread usamos precios que representan una ejecución conservadora:

- Al **comprar** la call de strike 535, pagamos su **ask**: \(18.19\).
- Al **vender** la call de strike 540, recibimos su **bid**: \(15.10\).

Por tanto, el débito inicial del spread, por acción, es:

\[
D = \operatorname{ask}(K_1) - \operatorname{bid}(K_2)
\]

\[
D = 18.19 - 15.10 = 3.09
\]

| Posición inicial | Efecto por acción |
|---|---:|
| Comprar call \(K_1=535\) | \(-18.19\) |
| Vender call \(K_2=540\) | \(+15.10\) |
| Débito neto del spread | **\(-3.09\)** |
| Comprar únicamente la call \(K_1=535\) | **\(-18.19\)** |

El spread requiere pagar **3.09 por acción**, frente a **18.19 por acción** al comprar únicamente la call de strike 535. En un contrato estándar de 100 acciones, equivaldrían a 309 y 1,819 antes de comisiones, respectivamente.

La call vendida reduce el desembolso inicial en 15.10 por acción. Esa reducción no es gratuita: en el siguiente paso veremos que limita la ganancia cuando SPY termine por encima de 540.

In [3]:
# Paso 2: coste de entrada usando cotizaciones ejecutables.
# Se paga el ask de la call comprada y se recibe el bid de la call vendida.

call_larga = spread_calls.iloc[0]   # K1 = 535
call_corta = spread_calls.iloc[1]   # K2 = 540

coste_call_sola = float(call_larga["ask"])
credito_call_vendida = float(call_corta["bid"])
debito_spread = coste_call_sola - credito_call_vendida

resumen_coste = pd.DataFrame(
    {
        "concepto": [
            "Ask call comprada (K1)",
            "Bid call vendida (K2)",
            "Débito neto del spread",
            "Coste de comprar solo la call K1",
        ],
        "importe_por_accion": [
            coste_call_sola,
            credito_call_vendida,
            debito_spread,
            coste_call_sola,
        ],
        "importe_por_contrato_100": [
            coste_call_sola * 100,
            credito_call_vendida * 100,
            debito_spread * 100,
            coste_call_sola * 100,
        ],
    }
)

display(resumen_coste.style.format({
    "importe_por_accion": "${:,.2f}",
    "importe_por_contrato_100": "${:,.2f}",
}))

print(
    f"El spread cuesta ${debito_spread:.2f} por acción, "
    f"frente a ${coste_call_sola:.2f} por comprar solo la call K1."
)

,concepto,importe_por_accion,importe_por_contrato_100
0,Ask call comprada (K1),$18.19,"$1,819.00"
1,Bid call vendida (K2),$15.10,"$1,510.00"
2,Débito neto del spread,$3.09,$309.00
3,Coste de comprar solo la call K1,$18.19,"$1,819.00"


El spread cuesta $3.09 por acción, frente a $18.19 por comprar solo la call K1.


## Paso 3 — Comparar resultados para precios hipotéticos al vencimiento

Ahora no usamos precios futuros reales. Elegimos escenarios hipotéticos para distinguir entre:

- el valor intrínseco de las calls al vencimiento;
- el resultado neto de la posición después del débito inicial;
- la diferencia entre el spread y comprar únicamente la call de strike 535.

Usaremos el débito calculado en el paso 2:

$ D = 3.09 $

Para el spread:

$  \Pi_{\text{spread}}(S_T) = \max(S_T-535,0) - \max(S_T-540,0) - 3.09 $

Para comprar únicamente la call de strike bajo:

$  \Pi_{\text{call}}(S_T) = \max(S_T-535,0)-18.19 $


| Escenario al vencimiento | $ S_T $ | Resultado spread | Resultado call sola | Lectura |
|---|---:|---:|---:|---|
| Por debajo de $ K_1 $ | 530 | -3.09 | -18.19 | Ambas calls expiran sin valor; el spread pierde solo su débito. |
| Entre strikes | 537 | -1.09 | -16.19 | La call comprada ya tiene valor intrínseco, pero todavía no recupera completamente el coste inicial. |
| Por encima de $ (K_2) $ | 545 | 1.91 | -8.19 | La call vendida empieza a restar valor; el spread ya alcanzó su ganancia máxima. |

Cuando $ S_T > 540 $, cada dólar adicional que gana la call comprada es compensado por un dólar adicional perdido en la call vendida. Por eso el resultado del spread queda fijado en:

$ (540-535)-3.09=1.91 $

La call comprada sola necesita una subida mucho mayor para recuperar su prima de 18.19, pero no tiene un límite superior de ganancia.